# How much can you say with rank 8?

**Topic:** LoRA &nbsp;·&nbsp; **Difficulty:** Medium

LoRA replaces a full weight update `dW` with a product `B A` of rank `r`, cutting
trainable parameters by orders of magnitude. The bet is that the update a
fine-tuning task needs is *intrinsically* low rank, and that bet is checkable.

Take a real full-rank update, find the best rank-`r` approximation for a range of
`r`, and see how much of it survives.

## The data (created in the next cell)

- `W0` `(768, 768)`: a pretrained weight.
- `dW_task` `(768, 768)`: the full-rank update a task actually needs, built to
  be dominated by a few directions.
- `ranks = [1, 2, 4, 8, 16, 32, 64, 128]`.

## Your task

With **NumPy only**:

1. For each rank `r`, compute the optimal rank-`r` approximation of `dW_task` by
   truncated SVD and record `captured[r] = 1 - ||dW - dW_r||_F / ||dW||_F`.
2. `params[r]`: LoRA's trainable parameter count `r * (768 + 768)`.
3. `full_params`: `768 * 768`.
4. `rank_for_95`: the smallest rank capturing at least 95%.
5. Set `answers` to a dict with `"captured"` (length-8), `"params"` (length-8),
   `"full_params"`, `"rank_for_95"`, `"param_saving_at_95"` =
   `full_params / params[rank_for_95]`.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks capture is monotone in rank and that the 95% rank saves at least 5x the
parameters.

In [ ]:
import numpy as np

rng = np.random.default_rng(4005)
d = 768
W0 = rng.normal(0, 1 / np.sqrt(d), (d, d))

U = np.linalg.qr(rng.normal(0, 1, (d, d)))[0]
V = np.linalg.qr(rng.normal(0, 1, (d, d)))[0]
spec = np.concatenate([np.linspace(1.0, 0.35, 24),
                       0.02 * np.exp(-np.arange(d - 24) / 90)])
dW_task = (U * spec) @ V.T
ranks = [1, 2, 4, 8, 16, 32, 64, 128]
print(f"W0 {W0.shape}, update rank profile: top-24 directions dominate")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("captured", [8], 4.179040729123814, 0.865363103723463, 1e-06),
    ("params", [8], 391680.0, -61878.92263912935, 1e-06),
    ("full_params", [], 589824.0, -98758.95964994715, 1e-06),
    ("rank_for_95", [], 32.0, -5.358016474063973, 1e-06),
    ("param_saving_at_95", [], 12.0, -2.00925617777399, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "captured" in answers:
    _c = np.asarray(answers["captured"], dtype=float)
    _mono = bool(np.all(np.diff(_c) >= -1e-12))
    _save = answers["param_saving_at_95"] >= 5
    print(f"  {'capture rises with rank':<24} {'ok' if _mono else 'wrong'}")
    print(f"  {'95% saves 5x params':<24} {'ok' if _save else 'wrong'}  "
          f"(rank {int(answers['rank_for_95'])}, {answers['param_saving_at_95']:.1f}x)")
    _ok = _ok and _mono and _save

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

Eckart-Young makes the truncated SVD the best possible rank-`r` approximation, so
this curve is an upper bound on what *any* rank-`r` method can capture, LoRA
included, and LoRA does not even get to see `dW_task` in advance. If the curve
were flat, LoRA could not work at all. The parameter saving is the other half:
rank 8 on a 768x768 matrix is 12,288 parameters against 589,824, and that ratio
is what makes fine-tuning a large model on one GPU possible.

```python
import numpy as np

Uu, s, Vt = np.linalg.svd(dW_task, full_matrices=False)
total = float(np.linalg.norm(dW_task))

captured = []
for r in ranks:
    approx = (Uu[:, :r] * s[:r]) @ Vt[:r]
    captured.append(1.0 - float(np.linalg.norm(dW_task - approx)) / total)
captured = np.array(captured)

params = np.array([r * (d + d) for r in ranks], dtype=float)
full_params = float(d * d)

idx = np.flatnonzero(captured >= 0.95)
r95 = int(ranks[int(idx[0])]) if len(idx) else -1
saving = full_params / (r95 * (d + d)) if r95 > 0 else 0.0

answers = {
    "captured": captured,
    "params": params,
    "full_params": full_params,
    "rank_for_95": float(r95),
    "param_saving_at_95": float(saving),
}
```